# Fitness coach — Google Colab (free GPU)

1. **Runtime → Change runtime type →** set **Hardware accelerator** to **GPU** (T4/L4 when available).
2. Run cells top to bottom.

**Data:** Upload this repo + `results/` to Drive, or clone a **public** GitHub repo. For **private** repos use a [fine-grained token](https://github.com/settings/tokens) in Colab **Secrets** (`GITHUB_TOKEN`) or paste a one-time clone URL.

In [ ]:
# Verify GPU (expect a CUDA device name)
!nvidia-smi

import torch
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

## 1) Get the code

Pick **one** path: clone from GitHub, or open from Google Drive after zipping the project locally.

In [ ]:
import os
import subprocess
from pathlib import Path

# --- Option A: clone (set your repo URL). Private repo: use token in URL or Colab Secrets. ---
REPO_URL = "https://github.com/YOUR_USER/Finess-coach-capstone-1.git"  # change me
BRANCH = "main"

WORK = Path("/content/fitness-coach")
if not WORK.is_dir():
    subprocess.run(
        ["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(WORK)],
        check=True,
    )
else:
    print("Already cloned:", WORK)

os.chdir(WORK)
print("cwd:", os.getcwd())

# --- Option B: Drive (comment Option A block above and use this instead) ---
# from google.colab import drive
# drive.mount("/content/drive")
# os.chdir("/content/drive/MyDrive/path/to/Finess-coach-capstone-1")

In [ ]:
# Install dependencies. Colab GPU images ship torch+CUDA; if cuda becomes False after this,
# reinstall wheels from https://pytorch.org/get-started/locally/ (pick the Colab CUDA version).
%pip install -q -r requirements.txt
%pip install -q -e .

# Optional: Kaggle / Riccio download inside Colab
# %pip install -q kagglehub

import torch
print("After install — cuda:", torch.cuda.is_available(), torch.__version__)

### Optional — build Riccio NPZs in Colab (MediaPipe, CPU)

**GPU does not speed up this step** here: MediaPipe runs on the **CPU**. Use **`--workers 0`** (default) so the script picks a pool from `os.cpu_count()` (often **2** on free Colab). For a large dataset, generating NPZs locally on a many-core machine is usually faster than Colab.

Uncomment and run **after** `kagglehub` is installed if you need `--download`.

In [ ]:
# Optional: uncomment to extract angles/NPZs in Colab (CPU + parallel workers)
# %pip install -q kagglehub opencv-python mediapipe
#
# import os
# import subprocess
# import sys
# from pathlib import Path
#
# os.chdir(WORK)
# out = Path("/content/fitness-coach/results/riccio_realtime_exercise_recognition")
# out.mkdir(parents=True, exist_ok=True)
# cmd = [
#     sys.executable,
#     "riccio_kaggle_video_pipeline.py",
#     "--download",
#     "--output-dir",
#     str(out),
#     "--output-stem",
#     "riccio_realtime_exercise_recognition",
#     "--workers",
#     "0",
#     "--skip-keypoints",
# ]
# print(" ".join(cmd))
# subprocess.run(cmd, cwd=str(WORK), check=True)

## 2) Point at your Riccio NPZs

Upload `results/riccio_realtime_exercise_recognition/` from your machine (or run the optional MediaPipe cell above), then set `KAGGLE_DIR` to the folder that contains `*_biomechanics.npz` and `*_labels.npz`.

In [ ]:
from pathlib import Path

# Folder with riccio_realtime_exercise_recognition_{biomechanics,labels,keypoints}.npz
KAGGLE_DIR = Path("/content/fitness-coach/results/riccio_realtime_exercise_recognition")
STEM = "riccio_realtime_exercise_recognition"
OUT_DIR = Path("/content/fitness-coach/results/exercise_bilstm_colab")

assert (KAGGLE_DIR / f"{STEM}_biomechanics.npz").is_file(), f"Missing NPZs under {KAGGLE_DIR}"
print("OK:", KAGGLE_DIR)

## 3) Train BiLSTM on GPU

Uses the same stratified **per-video** split when `labels.npz` includes `video_id`.

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    "train_exercise_bilstm.py",
    "--preset", "riccio",
    "--standardize",
    "--eval-test",
    "--epochs", "25",
    "--kaggle-angles-dir", str(KAGGLE_DIR),
    "--kaggle-stem", STEM,
    "--output-dir", str(OUT_DIR),
    "--kaggle-seed", "42",
]
print(" ".join(cmd))
r = subprocess.run(cmd, cwd=str(WORK))
raise SystemExit(r.returncode)

## 4) Download checkpoints / metrics

Run in a new code cell:

`from google.colab import files`  
`files.download("/content/fitness-coach/results/exercise_bilstm_colab/exercise_bilstm_best.pt")`

Or copy `results/exercise_bilstm_colab/` to Google Drive and sync locally.